# 🤗 x 🦾: Training SmolVLA with LeRobot Notebook

Welcome to the **LeRobot SmolVLA training notebook**! This notebook provides a ready-to-run setup for training imitation learning policies using the [🤗 LeRobot](https://github.com/huggingface/lerobot) library.

In this example, we train an `SmolVLA` policy using a dataset hosted on the [Hugging Face Hub](https://huggingface.co/), and optionally track training metrics with [Weights & Biases (wandb)](https://wandb.ai/).

## ⚙️ Requirements
- A Hugging Face dataset repo ID containing your training data (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- Optional: A [wandb](https://wandb.ai/) account if you want to enable training visualization
- Recommended: GPU runtime (e.g., NVIDIA A100) for faster training

## ⏱️ Expected Training Time
Training with the `SmolVLA` policy for 20,000 steps typically takes **about 5 hours on an NVIDIA A100** GPU. On less powerful GPUs or CPUs, training may take significantly longer!

## Example Output
Model checkpoints, logs, and training plots will be saved to the specified `--output_dir`. If `wandb` is enabled, progress will also be visualized in your wandb project dashboard.


## Install conda
This cell uses `condacolab` to bootstrap a full Conda environment inside Google Colab.


In [4]:
!pip install -q condacolab
import condacolab


RuntimeError: This module must ONLY run as part of a Colab notebook!

## Install LeRobot
This cell clones the `lerobot` repository from Hugging Face, installs FFmpeg (version 7.1.1), and installs the package in editable mode.


In [6]:
!git clone https://github.com/huggingface/lerobot.git
!conda install -y ffmpeg=7.1.1 -c conda-forge
!cd lerobot && pip install -e .

fatal: destination path 'lerobot' already exists and is not an empty directory.
2 channel Terms of Service accepted
Unexpected error writing token file:
  path: /home/icub/.conda/aau_token_host
  exception: write() argument must be str, not None
Channels:
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 26.1.1
    latest version: 26.7.1

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/icub/miniconda3

  added / updated specs:
    - ffmpeg=7.1.1


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    ffmpeg-7.1.1               | gpl_hbbdf940_911        10.0 MB  conda-forge
    libglib-2.86.2             |       h32235b2_0         3.8 MB  conda-forge
    libopencv-4.12.0           |qt6_py313h64bffa8_603        31.2 MB

## Weights & Biases login
Pega tu API key de [wandb.ai/authorize](https://wandb.ai/authorize) directamente en el comando. Si no quieres usar wandb, omite esta celda y asegúrate de tener `--wandb.enable=false` en el entrenamiento.


In [8]:
# Reemplaza TU_WANDB_API_KEY con tu clave de https://wandb.ai/authorize
# !wandb login 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
#!wandb login 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 miguel

!wandb login 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 #camada-naxo

wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/icub/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## Install SmolVLA dependencies

In [9]:
!cd lerobot && pip install -e ".[smolvla]"

Obtaining file:///home/icub/Desktop/mkloviiin/PMM_Final/lerobot
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Using cached num2words-0.5.14-py3-none-any.whl.metadata (13 kB)
  Using cached docopt-0.6.2.tar.gz (25 kB)
  Preparing metadata (setup.py) ... done
Using cached num2words-0.5.14-py3-none-any.whl (163 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 34.2 MB/s eta 0:00:00 0:00:01
  Building editable for lerobot (pyproject.toml) ... done
  Created wheel for lerobot: filename=lerobot-0.6.2-0.editable-py3-none-any.whl size=14689 sha256=15db432f0370f941d2e596587b11ee7e76524729952d62ab1a13fae96139c5d0
  Stored in directory: /tmp/pip-ephem-wheel-cache-yr_y1t0j/wheels/37/51/30/3ef0cc52b57277cd8f11994ab71f0bc2026cea17dae8922533
  DEPRECATION: Building 'docopt' using the legacy setup.py bdist_wheel mechanism, which 

### (Opcional) Filtrar features del dataset antes de entrenar

Los datasets nuevos graban con las features **ya ordenadas**: las 83 dimensiones de entrenamiento van primero, el resto después.

| Dims | Feature |
|---|---|
| `[0 – 21]` | Joint positions (22) |
| `[22– 43]` | Joint velocities (22) |
| `[44– 57]` | EEF pose rh + lh (14) |
| `[58– 76]` | VR targets (19) |
| `[77– 82]` | Touch contact (6) |
| **`[0 – 82]` → 83 dims de entrenamiento** | |
| `[83– 89]` | Object pose (7) |
| `[90–111]` | Joint torques (22) |
| `[112–117]` | External forces (6) |
| **118 dims total** | |

Cámaras: solo `frontview` + `head_cam` (se elimina `head_cam_track_hand`).

> Esta celda crea una copia del dataset con solo los primeros 83 dims y 2 cámaras.  
> Si prefieres entrenar con todo, salta esta celda y usa `max_state_dim=118`.

In [ ]:
import json
import shutil
import numpy as np
from pathlib import Path
import pandas as pd

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("/home/icub/mujoco_ws/REPO_ICUB/data/local_BlockLifting_2026-03-11_17-47-30")
OUTPUT_ROOT  = DATASET_ROOT.parent / (DATASET_ROOT.name + "_filtered")

# The dataset is recorded with features already in the right order:
# first 83 dims = training-relevant, rest = extra data.
STATE_INDICES = list(range(83))

# Cameras to keep (the rest are deleted from the filtered dataset)
CAMERAS_KEEP = {"observation.images.frontview", "observation.images.head_cam"}
# ────────────────────────────────────────────────────────────────────────────

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(DATASET_ROOT, OUTPUT_ROOT)

# ── info.json ────────────────────────────────────────────────────────────────
with open(OUTPUT_ROOT / "meta" / "info.json") as f:
    info = json.load(f)

orig_names = info["features"]["observation.state"]["names"]
new_names  = [orig_names[i] for i in STATE_INDICES]
info["features"]["observation.state"]["shape"] = [len(new_names)]
info["features"]["observation.state"]["names"] = new_names

for key in [k for k in list(info["features"]) if k.startswith("observation.images") and k not in CAMERAS_KEEP]:
    del info["features"][key]

with open(OUTPUT_ROOT / "meta" / "info.json", "w") as f:
    json.dump(info, f, indent=2)

# ── parquet data files ────────────────────────────────────────────────────────
for parquet_file in sorted((OUTPUT_ROOT / "data").glob("**/*.parquet")):
    df = pd.read_parquet(parquet_file)
    if "observation.state" in df.columns:
        df["observation.state"] = df["observation.state"].apply(
            lambda arr: np.asarray(arr)[:83].tolist()
        )
    for col in [c for c in df.columns if c.startswith("observation.images") and c not in CAMERAS_KEEP]:
        df = df.drop(columns=[col])
    df.to_parquet(parquet_file, index=False)

# ── stats.json ────────────────────────────────────────────────────────────────
stats_path = OUTPUT_ROOT / "meta" / "stats.json"
if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)
    if "observation.state" in stats:
        for stat_key, arr in stats["observation.state"].items():
            a = np.asarray(arr)
            if a.shape == (len(orig_names),):
                stats["observation.state"][stat_key] = a[:83].tolist()
    for key in [k for k in list(stats) if k.startswith("observation.images") and k not in CAMERAS_KEEP]:
        del stats[key]
    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

# ── remove unwanted video folders ────────────────────────────────────────────
videos_dir = OUTPUT_ROOT / "videos"
if videos_dir.exists():
    for cam_dir in list(videos_dir.iterdir()):
        if cam_dir.is_dir() and cam_dir.name not in CAMERAS_KEEP:
            shutil.rmtree(cam_dir)

print(f"Filtered dataset → {OUTPUT_ROOT}")
print(f"State: {len(orig_names)} → {len(new_names)} dims")
print(f"State features kept: {new_names}")

Filtered dataset → /home/icub/mujoco_ws/REPO_ICUB/data/local_BlockLifting_2026-03-17_16-32-35_filtered
State: 118 → 83 dims
State features kept: ['torso_pitch', 'torso_roll', 'torso_yaw', 'r_shoulder_pitch', 'r_shoulder_roll', 'r_shoulder_yaw', 'r_elbow', 'r_wrist_prosup', 'r_wrist_pitch', 'r_wrist_yaw', 'r_hand_finger', 'r_thumb_oppose', 'r_thumb_proximal', 'r_thumb_distal', 'r_index_proximal', 'r_index_distal', 'r_middle_proximal', 'r_middle_distal', 'r_pinky', 'neck_pitch', 'neck_roll', 'neck_yaw', 'torso_pitch_vel', 'torso_roll_vel', 'torso_yaw_vel', 'r_shoulder_pitch_vel', 'r_shoulder_roll_vel', 'r_shoulder_yaw_vel', 'r_elbow_vel', 'r_wrist_prosup_vel', 'r_wrist_pitch_vel', 'r_wrist_yaw_vel', 'r_hand_finger_vel', 'r_thumb_oppose_vel', 'r_thumb_proximal_vel', 'r_thumb_distal_vel', 'r_index_proximal_vel', 'r_index_distal_vel', 'r_middle_proximal_vel', 'r_middle_distal_vel', 'r_pinky_vel', 'neck_pitch_vel', 'neck_roll_vel', 'neck_yaw_vel', 'rh_ee_pos_x', 'rh_ee_pos_y', 'rh_ee_pos_z',

#### Merge two datasets

#### Remove features (using lerobot function)

In [6]:
keys_to_delete = ['l_shoulder_pitch', 'l_shoulder_roll', 'l_shoulder_yaw', 
                   'l_elbow', 'l_wrist_prosup', 
                   'l_wrist_pitch', 'l_wrist_yaw', 'l_hand_finger', 
                   'l_thumb_oppose', 'l_thumb_proximal', 'l_thumb_distal', 
                   'l_index_proximal', 'l_index_distal', 'l_middle_proximal', 
                   'l_middle_distal', 'l_pinky', 'l_shoulder_pitch_vel', 
                   'l_shoulder_roll_vel', 'l_shoulder_yaw_vel', 'l_elbow_vel', 
                   'l_wrist_prosup_vel', 'l_wrist_pitch_vel', 'l_wrist_yaw_vel', 
                   'l_hand_finger_vel', 'l_thumb_oppose_vel', 'l_thumb_proximal_vel', 
                   'l_thumb_distal_vel', 'l_index_proximal_vel', 'l_index_distal_vel', 
                   'l_middle_proximal_vel', 'l_middle_distal_vel', 'l_pinky_vel', 
                   'l_shoulder_pitch_torque', 'l_shoulder_roll_torque', 'l_shoulder_yaw_torque', 
                   'l_elbow_torque', 'l_wrist_prosup_torque', 'l_wrist_pitch_torque', 
                   'l_wrist_yaw_torque', 'l_hand_finger_torque', 'l_thumb_oppose_torque', 
                   'l_thumb_proximal_torque', 'l_thumb_distal_torque', 'l_index_proximal_torque', 
                   'l_index_distal_torque', 'l_middle_proximal_torque', 'l_middle_distal_torque', 
                   'l_pinky_torque', 'lh_ee_pos_x', 'lh_ee_pos_y', 
                   'lh_ee_pos_z', 'lh_ee_quat_w', 'lh_ee_quat_x', 
                   'lh_ee_quat_y', 'lh_ee_quat_z', 'lh_ext_force_x', 
                   'lh_ext_force_y', 'lh_ext_force_z', 'lh_touch_thumb', 
                   'lh_touch_index', 'lh_touch_middle'
                   ]

In [3]:
from pathlib import Path
import json
import shutil
import numpy as np
import pandas as pd

DATA_DIR = Path("/home/icub/mujoco_ws/REPO_ICUB/data")
DS1_ROOT = DATA_DIR / "local_icub_mujoco_demo_2026-03-04_11-30-07"
OUTPUT_ROOT = DATA_DIR / "local_icub_mujoco_demo_2026-03-04_11-30-07_left_arm_removed"

if "keys_to_delete" not in globals():
    raise RuntimeError("Primero ejecuta la celda anterior que define keys_to_delete")

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(DS1_ROOT, OUTPUT_ROOT)

# ── info.json: calcular índices a mantener ──────────────────────────────────
info_path = OUTPUT_ROOT / "meta" / "info.json"
with open(info_path) as f:
    info = json.load(f)

orig_names = info["features"]["observation.state"]["names"]
keys_to_delete_set = set(keys_to_delete)
keep_indices = [i for i, name in enumerate(orig_names) if name not in keys_to_delete_set]
new_names = [orig_names[i] for i in keep_indices]
deleted_names = [name for name in orig_names if name in keys_to_delete_set]
missing_in_state = sorted(keys_to_delete_set - set(orig_names))

info["features"]["observation.state"]["shape"] = [len(new_names)]
info["features"]["observation.state"]["names"] = new_names

with open(info_path, "w") as f:
    json.dump(info, f, indent=2)

# ── parquet: recortar observation.state según keep_indices ──────────────────
for parquet_file in sorted((OUTPUT_ROOT / "data").glob("**/*.parquet")):
    df = pd.read_parquet(parquet_file)
    if "observation.state" in df.columns:
        df["observation.state"] = df["observation.state"].apply(
            lambda arr: np.asarray(arr)[keep_indices].tolist()
        )
    df.to_parquet(parquet_file, index=False)

# ── stats.json: recortar vectores de estadísticas del estado ───────────────
stats_path = OUTPUT_ROOT / "meta" / "stats.json"
if stats_path.exists():
    with open(stats_path) as f:
        stats = json.load(f)

    if "observation.state" in stats:
        for stat_key, arr in list(stats["observation.state"].items()):
            a = np.asarray(arr)
            if a.ndim == 1 and a.shape[0] == len(orig_names):
                stats["observation.state"][stat_key] = a[keep_indices].tolist()

    with open(stats_path, "w") as f:
        json.dump(stats, f, indent=2)

print(f"Dataset original : {DS1_ROOT}")
print(f"Dataset filtrado : {OUTPUT_ROOT}")
print(f"State dims       : {len(orig_names)} -> {len(new_names)}")
print(f"Keys eliminadas  : {len(deleted_names)}")
if deleted_names:
    print("Primeras keys eliminadas:", deleted_names[:10])
if missing_in_state:
    print(f"⚠ keys_to_delete no encontradas en observation.state ({len(missing_in_state)}):")
    print(missing_in_state[:20])


Dataset loaded from: /home/icub/mujoco_ws/REPO_ICUB/data/local_icub_mujoco_demo_2026-03-04_11-30-07
Repo ID: local/icub_blocklifting_full_ok
Episodes: 50 | Frames: 35583
Features: ['action', 'observation.state', 'observation.images.head_cam', 'observation.images.head_cam_track_hand', 'observation.images.frontview', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']
State feature shape: (147,)
State feature names: ['torso_pitch', 'torso_roll', 'torso_yaw', 'l_shoulder_pitch', 'l_shoulder_roll', 'l_shoulder_yaw', 'l_elbow', 'l_wrist_prosup', 'l_wrist_pitch', 'l_wrist_yaw', 'l_hand_finger', 'l_thumb_oppose', 'l_thumb_proximal', 'l_thumb_distal', 'l_index_proximal', 'l_index_distal', 'l_middle_proximal', 'l_middle_distal', 'l_pinky', 'r_shoulder_pitch', 'r_shoulder_roll', 'r_shoulder_yaw', 'r_elbow', 'r_wrist_prosup', 'r_wrist_pitch', 'r_wrist_yaw', 'r_hand_finger', 'r_thumb_oppose', 'r_thumb_proximal', 'r_thumb_distal', 'r_index_proximal', 'r_index_distal', 'r_middle_prox

Copying observation.images.frontview videos: 100%|██████████| 1/1 [00:00<00:00, 48.77it/s]


#### Remove features (manually)

In [ ]:
import json
import shutil
import numpy as np
from pathlib import Path
import pandas as pd

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATASET_ROOT = Path("/home/icub/mujoco_ws/REPO_ICUB/data/local_icub_mujoco_demo_2026-03-04_11-30-07")
OUTPUT_ROOT  = DATASET_ROOT.parent / (DATASET_ROOT.name + "_filtered")

if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
shutil.copytree(DATASET_ROOT, OUTPUT_ROOT)

# ── parquet data files ────────────────────────────────────────────────────────
for parquet_file in sorted((OUTPUT_ROOT / "data").glob("**/*.parquet")):
    df = pd.read_parquet(parquet_file)
    if "observation.state" in df.columns:
        df["observation.state"] = df["observation.state"].apply(
            # From State feature names, delete all dims related to keys_to_delete
            lambda arr: np.asarray(arr)[:83].tolist()
        )
    # for col in [c for c in df.columns if c.startswith("observation.images") and c not in CAMERAS_KEEP]:
    #     df = df.drop(columns=[col])
    # df.to_parquet(parquet_file, index=False)

## Start training SmolVLA with LeRobot

This cell runs the `train.py` script from the `lerobot` library to train a robot control policy.  

Make sure to adjust the following arguments to your setup:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`:  
   Replace this with the Hugging Face Hub repo ID where your dataset is stored, e.g., `pepijn223/il_gym0`.

2. `--batch_size=64`: means the model processes 64 training samples in parallel before doing one gradient update. Reduce this number if you have a GPU with low memory.

3. `--output_dir=outputs/train/...`:  
   Directory where training logs and model checkpoints will be saved.

4. `--job_name=...`:  
   A name for this training job, used for logging and Weights & Biases.

5. `--policy.device=cuda`:  
   Use `cuda` if training on an NVIDIA GPU. Use `mps` for Apple Silicon, or `cpu` if no GPU is available.

6. `--wandb.enable=true`:  
   Enables Weights & Biases for visualizing training progress. You must be logged in via `wandb login` before running this.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Apunta al dataset curado local
DATASET_ROOT = "/home/icub/Desktop/mkloviiin/PMM_Final/data_curated/local_prueba_curated_2026-09-03_23-01-02"

!cd lerobot && PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/lerobot/scripts/lerobot_train.py \
  --policy.type=smolvla \
  --policy.vlm_model_name="HuggingFaceTB/SmolVLM2-500M-Video-Instruct" \
  --policy.load_vlm_weights=true \
  --policy.use_amp=true \
  --policy.actions_per_chunk=60 \
  --dataset.repo_id="local/prueba/curated" \
  --dataset.root={DATASET_ROOT} \
  --policy.max_state_dim=97 \
  --policy.push_to_hub=false \
  --batch_size=22 \
  --num_workers=32 \
  --steps=20000 \
  --log_freq=30 \
  --output_dir="/home/icub/Desktop/mkloviiin/PMM_Final/output/train/blockslifting" \
  --job_name="icub_smolvla_blocklifting" \
  --policy.device="cuda" \
  --wandb.enable=true \
  --wandb.project="icub_smolvla_blocklifting"


Traceback (most recent call last):
  File "/home/icub/Desktop/mkloviiin/PMM_Final/lerobot/src/lerobot/scripts/lerobot_train.py", line 48, in <module>
    from lerobot.common.train_utils import (
ModuleNotFoundError: No module named 'lerobot.common'


## Login into Hugging Face Hub
Now after training is done login into the Hugging Face hub and upload the last checkpoint

In [4]:
from getpass import getpass

hf_token = getpass("HF token: ")
!hf auth login --token "$hf_token"

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: write).
The token `Upload` has been saved to /home/icub/.cache/huggingface/stored_tokens
Your token has been saved to /home/icub/.cache/huggingface/token
Login successful.
The current active token is: `Upload`


In [5]:
!hf repo create CAMADA-UTFSM/icub_smolvla_blockstacking_v1 --type model --exist-ok
!hf upload CAMADA-UTFSM/icub_smolvla_blockstacking_v1 \
  /home/icub/mujoco_ws/REPO_ICUB/output/train/blockstacking20k_v1/checkpoints/last/pretrained_model \
  --repo-type model

Successfully created CAMADA-UTFSM/icub_smolvla_blockstacking_v1 on the Hub.
Your repo is now available at https://huggingface.co/CAMADA-UTFSM/icub_smolvla_blockstacking_v1
Start hashing 7 files.
Finished hashing 7 files.
Processing Files (0 / 0)      : |                  |  0.00B /  0.00B            
New Data Upload               : |                  |  0.00B /  0.00B            

  ...d_model/model.safetensors:   0%|              |  603kB /  907MB            

Processing Files (0 / 1)      :   0%|              |  603kB /  907MB,  337kB/s  
New Data Upload               :   1%|▏             |  603kB / 67.1MB,  337kB/s  

Processing Files (0 / 1)      :   0%|              | 1.21MB /  907MB,  606kB/s  
New Data Upload               :   2%|▎             | 1.21MB / 67.1MB,  606kB/s  

Processing Files (0 / 1)      :   0%|              | 1.81MB /  907MB,  826kB/s  
New Data Upload               :   1%|▏             | 1.81MB /  134MB,  826kB/s  

Processing Files (0 / 1)      :   0%|        